# Telling specifications apart

The answers to Section 8 of the sample questions, `documentation/tex/questions`, run.
The questions concern the exponential-kernel Hawkes processes of Section 1.3.3 of the lecture
notes, their stability and their clusters, Section 1.3.4, and what a path drawn from one shows
of its specification.
The two types are read as the market buys (component 0) and the market sells (component 1).

Each question's code is copied here verbatim, wrapped in a function where it prints.
`outcome` returns what a call prints, or the name of the exception it raises.
`verify` evaluates every choice of the question and asserts that the key holds and that no
other choice does.
A Monte Carlo value is near a number when it lies within four standard errors of it; the
standard errors are printed beside the values, and the numbers a choice offers lie many
standard errors apart.

In [ ]:
import contextlib
import io
import time
import timeit
from itertools import islice
from types import SimpleNamespace

import numpy as np
import pandas as pd
from scipy.linalg import expm

from unito26.lob import config
from unito26.lob.hawkes import ExponentialHawkes, HawkesParams
from unito26.lob.messages import (
    BUY, SELL, GridDepth, limit_order, market_order, withdrawal,
)
from unito26.lob.orderbook import AggregateBook

from unito26.lob.messages import BranchingRatio, Decay


def outcome(function, *arguments):
    """What a call prints, or the name of the exception it raises."""
    printed = io.StringIO()
    try:
        with contextlib.redirect_stdout(printed):
            function(*arguments)
    except Exception as error:
        return type(error).__name__
    return printed.getvalue().strip()


def verify(label, key, holds):
    """Assert that, of the choices of a question, the one lettered `key` holds and no other does."""
    holding = [letter for letter, held in zip("ABCDE", holds) if held]
    assert holding == [key], f"{label}: {holding or 'no choice'} hold, the key is {key}"
    print(f"{label}: ({key}) holds, and no other choice does")


def lines(printed):
    """The printed lines, each with its runs of spaces made one."""
    return [" ".join(line.split()) for line in printed.split("\n")]


def near(value, target, error):
    """Within four standard errors."""
    return abs(value - target) < 4 * error


def spectrum(matrix):
    """The eigenvalues, real parts first, rounded."""
    return np.round(np.sort_complex(np.linalg.eigvals(np.asarray(matrix, dtype=float))), 4)

## `question.whichOrderFlowSpecificationsAreRefused`: the branching ratio, by the spectrum

The eigenvalues, the column sums and the row sums of each $\Gamma = A/4$: the misreadings that
the distractors encode read a column sum or a row sum above one as $\rho \ge 1$, read the
diagonal alone, or take the constructor to require every baseline positive.

In [ ]:
def run():
    specifications = {
        "S1": ([0.4, 0.1], [[0.8, 0.0], [4.0, 2.4]]),
        "S2": ([0.3, 0.3], [[2.4, 2.8], [0.8, 0.4]]),
        "S3": ([0.2, 0.2], [[2.0, 1.8], [1.6, 3.2]]),
        "S4": ([0.0, 0.5], [[1.2, 0.4], [0.4, 1.2]]),
    }
    refused = []
    for name, (baseline, excitation) in specifications.items():
        try:
            HawkesParams(baseline=baseline, excitation=excitation, decay=4.0)
        except ValueError:
            refused.append(name)
    print(refused)


for name, excitation in (
    ("S1", [[0.8, 0.0], [4.0, 2.4]]),
    ("S2", [[2.4, 2.8], [0.8, 0.4]]),
    ("S3", [[2.0, 1.8], [1.6, 3.2]]),
    ("S4", [[1.2, 0.4], [0.4, 1.2]]),
):
    gamma = np.array(excitation) / 4.0
    print(f"{name}: eigenvalues {spectrum(gamma)}, column sums {gamma.sum(axis=0)}, "
          f"row sums {gamma.sum(axis=1)}")
printed = outcome(run)
print(printed)
verify("question.whichOrderFlowSpecificationsAreRefused", "C", [
    printed == "[]",
    printed == "['S1', 'S3']",
    printed == "['S3']",
    printed == "['S3', 'S4']",
    printed == "['S1', 'S2', 'S3']",
])

## `question.fixedPointSolvedAtAndAboveCriticality`: the solve checks nothing

$1 - 2.0/4.0$ is exactly $0.5$, so the second matrix $I - \Gamma$ is exactly singular.

In [ ]:
assert 1 - 2.0 / 4.0 == 0.5


def run():
    def stationary(mu, excitation, beta):
        return np.linalg.solve(np.eye(2) - excitation / beta, mu)

    mu = np.array([0.2, 0.2])
    for excitation in ([[2.0, 1.2], [1.2, 2.0]],
                       [[2.0, 2.0], [2.0, 2.0]],
                       [[2.0, 2.4], [2.4, 2.0]]):
        try:
            print(stationary(mu, np.array(excitation), 4.0))
        except np.linalg.LinAlgError:
            print("singular")


printed = lines(outcome(run))
print(printed)
verify("question.fixedPointSolvedAtAndAboveCriticality", "A", [
    printed == ["[1. 1.]", "singular", "[-2. -2.]"],
    printed == ["[1. 1.]", "singular", "singular"],
    printed == ["[1. 1.]", "singular", "[2. 2.]"],
    printed == ["[1. 1.]", "[inf inf]", "[-2. -2.]"],
])

## `question.lookalikeKernelsOneSupercritical`: one entry of the kernel, and a path that explodes

The first specification is stationary, and the standard error of its count over $(0, T]$ is
$\sqrt{T\,\mathbf 1^\top C \mathbf 1}$, with $C$ the `count_covariance_rate` of the package.
The second is not; the time of its $10^5$-th event is printed for several seeds.

In [ ]:
def run():
    for excitation in ([[2.0, 0.4], [3.6, 2.0]], [[2.0, 1.6], [3.6, 2.0]]):
        params = SimpleNamespace(
            baseline=np.array([0.2, 0.2]),
            excitation=np.array(excitation),
            decay=4.0,
            dimension=2,
        )
        path = ExponentialHawkes(params, rng=1).events(4000.0)
        print(len(list(islice(path, 100_000))))


stationary_one = HawkesParams(baseline=[0.2, 0.2], excitation=[[2.0, 0.4], [3.6, 2.0]], decay=4.0)
print("spectra:", spectrum(np.array([[2.0, 0.4], [3.6, 2.0]]) / 4),
      spectrum(np.array([[2.0, 1.6], [3.6, 2.0]]) / 4))
print("fixed points:", stationary_one.stationary_intensity(),
      np.linalg.solve(np.eye(2) - np.array([[2.0, 1.6], [3.6, 2.0]]) / 4, [0.2, 0.2]))
expected = stationary_one.stationary_intensity().sum() * 4000.0
absolute = -np.linalg.solve(np.eye(2) - np.array([[2.0, 1.6], [3.6, 2.0]]) / 4, [0.2, 0.2]).sum() * 4000.0
error = np.sqrt(4000.0 * stationary_one.count_covariance_rate().sum())
supercritical = SimpleNamespace(
    baseline=np.array([0.2, 0.2]), excitation=np.array([[2.0, 1.6], [3.6, 2.0]]),
    decay=4.0, dimension=2,
)
for seed in range(1, 6):
    last = list(islice(ExponentialHawkes(supercritical, rng=seed).events(4000.0), 100_000))[-1]
    print(f"seed {seed}: the 100000th event of the second specification at t = {last[0]:.1f} s")

first, second = map(int, outcome(run).split("\n"))
print(f"printed {first} and {second}; expected {expected:.0f} +- {error:.0f}, "
      f"|fixed point| x T = {absolute:.0f}")
verify("question.lookalikeKernelsOneSupercritical", "D", [
    near(first, expected, error) and near(second, expected, error),
    near(first, expected, error) and near(second, absolute, error),
    first == second == 100_000,
    near(first, expected, error) and second == 100_000,
])

## `question.buysWithNeitherImmigrantsNorExciters`: a branching ratio of 2, and a stationary flow

The sells alone are a one-type Hawkes process with $\Gamma = 0.5$, of rate $0.5/(1-0.5) = 1$ and
count variance rate $\lambda^*/(1-\Gamma)^2 = 4$. The first generation alone, $\mu(1+\Gamma)$,
would give $0.75$.

In [ ]:
def run():
    params = SimpleNamespace(
        baseline=np.array([0.0, 0.5]),
        excitation=np.array([[8.0, 0.0], [2.0, 2.0]]),
        decay=4.0,
        dimension=2,
    )
    events = ExponentialHawkes(params, rng=1).events(1e4)
    types = [event_type for _, event_type in events]
    print((np.bincount(types, minlength=2) / 1e4).round(1))


print("spectrum of Gamma:", spectrum(np.array([[8.0, 0.0], [2.0, 2.0]]) / 4))
print(f"standard error of the rate of the sells: {np.sqrt(4 / 1e4):.3f}")
printed = outcome(run)
ended = True
print(printed)
verify("question.buysWithNeitherImmigrantsNorExciters", "D", [
    not ended,
    lines(printed) == ["[0. 0.5]"],
    lines(printed) == ["[0. 0.8]"],
    lines(printed) == ["[0. 1.]"],
])

## `question.balanceOfBuysAndSellsAgainstImmigrants`: the share of the buys, by hand

The share of the buys $N_0/(N_0 + N_1)$ over $T$ has, to first order, the variance
$g^\top C g / T$ with $g = (\lambda^*_1, -\lambda^*_0)/\bar\lambda^2$. The distractors are the
shares of the immigrants, of the first generation $\mu + \Gamma\mu$, and of the transposed
kernels.

In [ ]:
def run():
    for params in (config.asymmetric_pair_params(),
                   config.one_way_pair_params()):
        events = ExponentialHawkes(params, rng=5).events(2e4)
        types = np.array([event_type for _, event_type in events])
        print(round(np.mean(types == 0), 2))


def share_of_buys(params):
    stationary = params.stationary_intensity()
    total = stationary.sum()
    gradient = np.array([stationary[1], -stationary[0]]) / total**2
    error = np.sqrt(gradient @ params.count_covariance_rate() @ gradient / 2e4)
    return stationary[0] / total, error


def transposed(params):
    return HawkesParams(baseline=params.baseline, excitation=params.excitation.T,
                        decay=params.decay)


pairs = (config.asymmetric_pair_params(), config.one_way_pair_params())
for params in pairs:
    mu, gamma = params.baseline, params.branching_matrix
    first_generation = mu + gamma @ mu
    print(f"share of buys {share_of_buys(params)[0]:.4f} +- {share_of_buys(params)[1]:.4f}; "
          f"immigrants {mu[0] / mu.sum():.4f}; first generation "
          f"{first_generation[0] / first_generation.sum():.4f}; "
          f"transposed {share_of_buys(transposed(params))[0]:.4f}")
(asymmetric, a_error), (one_way, o_error) = (share_of_buys(params) for params in pairs)
a, o = map(float, outcome(run).split("\n"))
print("printed", a, o)
verify("question.balanceOfBuysAndSellsAgainstImmigrants", "B", [
    near(a, 0.25, a_error) and near(o, 0.8, o_error),
    near(a, 0.67, a_error) and near(o, 0.25, o_error),
    near(a, 0.57, a_error) and near(o, 0.46, o_error),
    near(a, 0.25, a_error) and near(o, 0.76, o_error),
])

## `question.endogenousFractionWeighsTheColumnSums`: the endogenous fraction against $\rho$

In [ ]:
def run():
    params = config.asymmetric_pair_params()
    gamma, mu = params.branching_matrix, params.baseline
    lam = params.stationary_intensity()
    print(round(params.branching_ratio, 4),
          round(1 - mu.sum() / lam.sum(), 4),
          round(gamma.sum(axis=0) @ lam / lam.sum(), 4),
          round(gamma.sum(axis=1) @ lam / lam.sum(), 4))


printed = outcome(run)
print(printed)
verify("question.endogenousFractionWeighsTheColumnSums", "C", [
    printed == "0.6 0.6 0.7333 1.1",
    printed == "0.6 0.7333 1.1 0.7333",
    printed == "0.6 0.7333 0.7333 1.1",
    printed == "ValueError",
    printed == "0.6 0.6 0.6 0.6",
])

## `question.clusterSizeByFounderAndItsTranspose`: the cluster of $\Gamma$ and that of $\Gamma^\top$

The expected sizes are the column sums of $(I - \Gamma)^{-1}$ for A and its row sums for B.
The standard errors are those of the same means on the same seed, recomputed.

In [ ]:
def mean_size(branching, founder, n, rng):
    generation = np.zeros((2, n), dtype=int)
    generation[founder] = 1
    sizes = np.ones(n, dtype=int)
    while generation.any():
        generation = rng.poisson(branching @ generation)
        sizes += generation.sum(axis=0)
    return sizes.mean()

mean_size_a = mean_size

In [ ]:
def mean_size(branching, founder, n, rng):
    generation = np.zeros((2, n), dtype=int)
    generation[founder] = 1
    sizes = np.ones(n, dtype=int)
    while generation.any():
        generation = rng.poisson(branching.T @ generation)
        sizes += generation.sum(axis=0)
    return sizes.mean()

mean_size_b = mean_size

In [ ]:
def run(mean_size):
    gamma = config.one_way_pair_params().branching_matrix
    rng = np.random.default_rng(4)
    print(round(mean_size(gamma, 0, 20_000, rng), 1),
          round(mean_size(gamma, 1, 20_000, rng), 1))


def sizes(branching, founder, n, rng):
    generation = np.zeros((2, n), dtype=int)
    generation[founder] = 1
    total = np.ones(n, dtype=int)
    while generation.any():
        generation = rng.poisson(branching @ generation)
        total += generation.sum(axis=0)
    return total


gamma = config.one_way_pair_params().branching_matrix
descendants = np.linalg.inv(np.eye(2) - gamma)
print("column sums of (I - Gamma)^-1:", descendants.sum(axis=0),
      " row sums:", descendants.sum(axis=1))
for name, branching in (("A", gamma), ("B", gamma.T)):
    rng = np.random.default_rng(4)
    errors = [sizes(branching, founder, 20_000, rng).std() / np.sqrt(20_000) for founder in (0, 1)]
    print(f"{name}: standard errors {np.round(errors, 3)}")
a, b = outcome(run, mean_size_a), outcome(run, mean_size_b)
print("A:", a, "   B:", b)
pair = lambda printed: [float(x) for x in printed.split()] if printed[0].isdigit() else None
close = lambda printed, x, y: pair(printed) is not None and abs(pair(printed)[0] - x) < 0.15 \
    and abs(pair(printed)[1] - y) < 0.15
ended = True
verify("question.clusterSizeByFounderAndItsTranspose", "D", [
    close(a, 2.5, 2.5) and close(b, 2.5, 2.5),
    close(a, 4.375, 2.5) and close(b, 4.375, 2.5),
    close(a, 1.25, 5.625) and close(b, 4.375, 2.5),
    close(a, 4.375, 2.5) and close(b, 1.25, 5.625),
    not ended,
])

## `question.balancedTargetOnTheAsymmetricShape`: a target the baseline cannot reach

The distractors: the column sums for the row sums, the shape scaled by $\rho$ rather than by
$\rho/0.6$, a constructor that checks only $\rho < 1$, and a shape left at its own radius.

In [ ]:
def run():
    shape = config.asymmetric_pair_params().excitation
    target = np.array([1.0, 1.0])
    for rho in (0.2, 0.3, 0.5):
        try:
            params = HawkesParams.from_stationary_intensity(
                target, shape, Decay(4.0), BranchingRatio(rho)
            )
            print(rho, params.baseline.round(2))
        except ValueError:
            print(rho, "refused")


shape_gamma = config.asymmetric_pair_params().branching_matrix
print("spectrum of the shape's Gamma:", spectrum(shape_gamma),
      " row sums:", shape_gamma.sum(axis=1), " column sums:", shape_gamma.sum(axis=0))
printed = lines(outcome(run))
print(printed)
verify("question.balancedTargetOnTheAsymmetricShape", "A", [
    printed == ["0.2 [0.5 0.9]", "0.3 [0.25 0.85]", "0.5 refused"],
    printed == ["0.2 [0.87 0.53]", "0.3 [0.8 0.3]", "0.5 refused"],
    printed == ["0.2 [0.5 0.9]", "0.3 [0.25 0.85]", "0.5 [-0.25 0.75]"],
    printed == ["0.2 [0.7 0.94]", "0.3 [0.55 0.91]", "0.5 [0.25 0.85]"],
    printed == ["0.2 refused", "0.3 refused", "0.5 refused"],
])

## `question.selfAndCrossPairsRelaxAtDifferentRates`: the spectrum of $K = A - \beta I$

In [ ]:
def run():
    for params in (config.self_exciting_pair_params(),
                   config.cross_exciting_pair_params()):
        rates = -np.linalg.eigvals(params.hurwitz_matrix).real
        print(np.sort(rates).round(2), round(params.relaxation_time(), 2))


for params in (config.self_exciting_pair_params(), config.cross_exciting_pair_params()):
    print("spectrum of Gamma:", spectrum(params.branching_matrix))
printed = lines(outcome(run))
print(printed)
verify("question.selfAndCrossPairsRelaxAtDifferentRates", "B", [
    printed == ["[1. 1.4] 1.0", "[1. 1.4] 1.0"],
    printed == ["[1. 1.4] 1.0", "[1. 6.6] 1.0"],
    printed == ["[1. 1.4] 0.71", "[1. 6.6] 0.15"],
    printed == ["[0.25 0.35] 4.0", "[0.25 1.65] 4.0"],
])

## `question.fanoFactorInShortAndLongWindows`: a window that holds one event, and one that holds clusters

In windows long against the relaxation time the Fano factor of the total count tends to
$1/(1-\rho)^2$ for both pairs, whose column sums are constant; at $w = 100$ it falls short of the
limit by about $\rho(2-\rho)/((1-\rho)^2 \beta (1-\rho) w)$. Its estimate over $n = 1000$ windows
has a relative standard error of about $\sqrt{2/n}$.

In [ ]:
def fano(times, window, horizon):
    counts = np.bincount((times // window).astype(int),
                         minlength=round(horizon / window))
    return counts.var() / counts.mean()

In [ ]:
def run():
    for params in (config.weakly_exciting_pair_params(),
                   config.self_exciting_pair_params()):
        events = ExponentialHawkes(params, rng=6).events(1e5)
        times = np.array([t for t, _ in events])
        print(round(fano(times, 0.01, 1e5), 2),
              round(fano(times, 100.0, 1e5), 1))


def long_window_fano(params, window):
    rho, beta = params.branching_ratio, params.decay
    shortfall = rho * (2 - rho) / (1 - rho) ** 2 / (beta * (1 - rho) * window)
    return 1 / (1 - rho) ** 2 - shortfall


weak_line, self_line = ([float(x) for x in line.split()] for line in outcome(run).split("\n"))
limits = [long_window_fano(params, 100.0) for params in
          (config.weakly_exciting_pair_params(), config.self_exciting_pair_params())]
errors = [limit * np.sqrt(2 / 1000) for limit in limits]
print("weakly exciting:", weak_line, "  self-exciting:", self_line)
print("at w = 100: expected", np.round(limits, 2), "+-", np.round(errors, 2))
one = lambda x: abs(x - 1) < 0.1
weak_long, self_long = (lambda x: near(x, limits[0], errors[0])), (lambda x: near(x, limits[1], errors[1]))
verify("question.fanoFactorInShortAndLongWindows", "B", [
    all(map(weak_long, weak_line)) and all(map(self_long, self_line)),
    one(weak_line[0]) and one(self_line[0]) and weak_long(weak_line[1]) and self_long(self_line[1]),
    weak_long(weak_line[0]) and self_long(self_line[0]) and one(weak_line[1]) and one(self_line[1]),
    all(map(one, weak_line + self_line)),
])

## `question.statisticsThatTellSelfFromCross`: what the types show and the times do not

Each statistic is recomputed on the ten consecutive blocks of $10^4$ seconds of the same path,
and its standard error is the standard deviation of the ten over $\sqrt{10}$. Two statistics
differ when the difference of the printed values exceeds eight standard errors of the
difference, and agree when it is within three.

In [ ]:
def statistics(params, seed):
    path = np.array(list(ExponentialHawkes(params, rng=seed).events(1e5)))
    times, types = path[:, 0], path[:, 1].astype(int)
    window = (times // 100).astype(int)
    buys = np.bincount(window[types == 0], minlength=1000)
    sells = np.bincount(window[types == 1], minlength=1000)
    total = buys + sells
    after_sell = types[1:][types[:-1] == 1]
    print(round(buys.sum() / 1e5, 2),
          round(total.var() / total.mean(), 1),
          round(np.corrcoef(buys, sells)[0, 1], 2),
          round(np.mean(after_sell == 1), 2))

In [ ]:
def run():
    statistics(config.self_exciting_pair_params(), 1)
    statistics(config.cross_exciting_pair_params(), 2)


def block_errors(params, seed):
    path = np.array(list(ExponentialHawkes(params, rng=seed).events(1e5)))
    times, types = path[:, 0], path[:, 1].astype(int)
    rows = []
    for block in range(10):
        kept = (times > 1e4 * block) & (times <= 1e4 * (block + 1))
        t, e = times[kept] - 1e4 * block, types[kept]
        window = (t // 100).astype(int)
        buys = np.bincount(window[e == 0], minlength=100)[:100]
        sells = np.bincount(window[e == 1], minlength=100)[:100]
        total = buys + sells
        after_sell = e[1:][e[:-1] == 1]
        rows.append([buys.sum() / 1e4, total.var() / total.mean(),
                     np.corrcoef(buys, sells)[0, 1], np.mean(after_sell == 1)])
    return np.std(rows, axis=0, ddof=1) / np.sqrt(10)


self_values, cross_values = (np.array([float(x) for x in line.split()])
                             for line in outcome(run).split("\n"))
error = np.hypot(block_errors(config.self_exciting_pair_params(), 1),
                 block_errors(config.cross_exciting_pair_params(), 2))
gap = np.abs(self_values - cross_values) / error
print("self-exciting: ", self_values, "\ncross-exciting:", cross_values)
print("differences in standard errors (rate, Fano, correlation, share):", np.round(gap, 1))
differ = lambda subset: all(gap[i] > 8 for i in subset) and all(
    gap[i] < 3 for i in range(4) if i not in subset)
verify("question.statisticsThatTellSelfFromCross", "C", [
    differ([]),
    differ([2]),
    differ([2, 3]),
    differ([1, 2, 3]),
    differ([3]),
])

## `question.stationaryIntensityBySeriesAgainstSolve`: a series of $1/(1-\rho)$ terms against one solve

$\mu = (0.5, 0.5)$ is a right Perron vector of every $\Gamma$ of the ladder, so the $k$-th term
is $\rho^k\mu$ and the series stops after about $\log(5 \times 10^{11}) / (1 - \rho)$ terms.
Each rung is timed nine times, in turn with the others, and the shortest time is kept.
A time is the processor time of this process, `time.process_time`, which other processes do not inflate.

In [ ]:
def by_series(mu, gamma, tolerance):
    total, term = mu.copy(), mu.copy()
    while np.abs(term).max() > tolerance:
        term = gamma @ term
        total = total + term
    return total

In [ ]:
def by_solve(mu, gamma):
    return np.linalg.solve(np.eye(len(mu)) - gamma, mu)

In [ ]:
mu = np.array([0.5, 0.5])
ladder = (0.9, 0.99, 0.999)
gammas = [rho * np.array([[0.7, 0.3], [0.3, 0.7]]) for rho in ladder]
for gamma in gammas:
    assert np.allclose(by_series(mu, gamma, 1e-12), by_solve(mu, gamma), rtol=1e-9)
clock = time.process_time
rounds = [
    [(timeit.timeit(lambda: by_series(mu, gamma, 1e-12), number=1, timer=clock),
      timeit.timeit(lambda: by_solve(mu, gamma), number=500, timer=clock) / 500)
     for gamma in gammas]
    for _ in range(9)
]
series_times = [min(times[k][0] for times in rounds) for k in range(3)]
solve_times = [min(times[k][1] for times in rounds) for k in range(3)]
for rho, series, solve in zip(ladder, series_times, solve_times):
    print(f"rho = {rho}: series {series * 1e6:9.0f} us   solve "
          f"{solve * 1e6:6.1f} us   ratio {series / solve:7.0f}")

series_growth = [series_times[i + 1] / series_times[i] for i in range(2)]
solve_growth = [solve_times[i + 1] / solve_times[i] for i in range(2)]
increments = [series_times[i + 1] - series_times[i] for i in range(2)]
print("growth from rung to rung: series", np.round(series_growth, 1),
      " solve", np.round(solve_growth, 2))
solve_faster = all(s / v > 5 for s, v in zip(series_times, solve_times))
verify("question.stationaryIntensityBySeriesAgainstSolve", "A", [
    solve_faster and all(5 < g < 20 for g in series_growth) and all(g < 2 for g in solve_growth),
    all(s < v for s, v in zip(series_times, solve_times)),
    solve_faster and all(g < 2 for g in series_growth + solve_growth),
    solve_faster and 0.5 < increments[1] / increments[0] < 2,
])